# Importations

In [1]:
# Numerical and scientific python programming
import numpy as np

# Local importations
from moments.bloch import compute_pauli_basis, compute_tensor_basis, compute_subset_index_map
from moments.optimization import optimize_moment_preserving_entanglement

# Optimization

In [2]:
N = 2
d = 2 ** N

pauli_basis = compute_pauli_basis()
local_bases = [pauli_basis.copy()] * N
local_basis_sizes = [len(basis) for basis in local_bases]

tensor_basis = compute_tensor_basis(local_bases)
subset_index_map = compute_subset_index_map(local_basis_sizes)

Rt = {(1,): 0.5, (2,): 0.5, (1, 2): 1}

In [3]:
optimization_result = optimize_moment_preserving_entanglement(d, tensor_basis, subset_index_map, Rt,
                                                              metric="concurrence", optimization="minimize", cholesky_opt=True)

## Validation

In [4]:
checks = optimization_result.checks

print("Is the output a valid density matrix?", checks["is_valid_dm"])

bloch_equal = []
bloch_diff = {}
moments_equal = []
for subset in optimization_result.bloch_initial.keys():
        bloch_equal.append(np.allclose(optimization_result.bloch_initial[subset], optimization_result.bloch_final[subset]))
        bloch_diff[subset] = float(np.linalg.norm(optimization_result.bloch_initial[subset] - optimization_result.bloch_final[subset]))
        moments_equal.append(np.allclose(optimization_result.moments_initial[subset], optimization_result.moments_final[subset]))

print("\nAre the density matrices equal?", np.allclose(optimization_result.rho_initial, optimization_result.rho_final))
print("Density matrix difference:", np.linalg.norm(optimization_result.rho_initial - optimization_result.rho_final))

print("\nAre the Bloch vectors equal?", all(bloch_equal))
print("Bloch vector difference:", bloch_diff)

print("\nAre the Bloch lengths equal?", checks["moments_equal"])
print("Bloch lengths difference?", checks["moments_distance"])


print("\nIs the metric equal?", np.isclose(optimization_result.metric_initial, optimization_result.metric_final))
print("Metric difference:", abs(optimization_result.metric_initial - optimization_result.metric_final))

Is the output a valid density matrix? True

Are the density matrices equal? False
Density matrix difference: 0.5617401764635401

Are the Bloch vectors equal? False
Bloch vector difference: {(1,): 0.3789616282347403, (2,): 0.5016300942144054, (1, 2): 0.9311087134794078}

Are the Bloch lengths equal? True
Bloch lengths difference? {(1,): 2.3314683517128287e-15, (2,): 4.3298697960381105e-15, (1, 2): 6.106226635438361e-15}

Is the metric equal? False
Metric difference: 0.5776442061141094


In [5]:
optimizer_info = optimization_result.optimizer_info
print("Result success:", optimizer_info["result_success"])
print("Result message:", optimizer_info["result_message"])

Result success: True
Result message: Optimization terminated successfully


In [6]:
print(optimization_result.metric_initial)
print(optimization_result.metric_final)

0.5776851297674555
4.0923653346118486e-05
